In [ ]:
!pip install scikit-learn pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/semantic-search-system'
os.makedirs(f'{BASE}/data/processed', exist_ok=True)
os.makedirs(f'{BASE}/data/raw', exist_ok=True)
print('Drive mounted and folders ready')

In [ ]:
import re
import pickle
import pandas as pd
from sklearn.datasets import fetch_20newsgroups

print('Fetching 20 Newsgroups dataset...')

newsgroups = fetch_20newsgroups(
    subset='all',
    remove=('headers', 'footers', 'quotes'),
    categories=[
        'sci.med', 'sci.space', 'sci.electronics', 'sci.crypt',
        'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware',
        'rec.autos', 'rec.sport.baseball', 'rec.sport.hockey',
        'talk.politics.guns', 'talk.politics.misc',
        'alt.atheism', 'soc.religion.christian', 'misc.forsale'
    ]
)

print(f'Total documents : {len(newsgroups.data)}')
print(f'Categories      : {newsgroups.target_names}')

In [ ]:
df_raw = pd.DataFrame({
    'text'    : newsgroups.data,
    'label'   : newsgroups.target,
    'category': [newsgroups.target_names[t] for t in newsgroups.target]
})
df_raw['length'] = df_raw['text'].str.len()
print(df_raw[['category','length']].head(5))
print('\nLength stats:')
print(df_raw['length'].describe())

In [ ]:
def clean_text(text):
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s.,!?\'-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df_raw['clean_text'] = df_raw['text'].apply(clean_text)
df_clean = df_raw[df_raw['clean_text'].str.len() > 50].copy().reset_index(drop=True)

print(f'Before cleaning : {len(df_raw)}')
print(f'After cleaning  : {len(df_clean)}')
print(f'Removed         : {len(df_raw) - len(df_clean)}')

In [ ]:
corpus = {
    'texts'       : df_clean['clean_text'].tolist(),
    'raw_texts'   : df_clean['text'].tolist(),
    'labels'      : df_clean['label'].tolist(),
    'categories'  : df_clean['category'].tolist(),
    'target_names': list(newsgroups.target_names)
}

save_path = f'{BASE}/data/processed/clean_corpus.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(corpus, f)

print(f'Saved to Google Drive: {save_path}')
print(f'Total documents saved: {len(corpus["texts"])}')